In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [2]:
from src_oop.core.database import Database
from src_oop.jobs.unit.queries import query_adv_spend
from src_oop.jobs.unit.config import unit_gs
from src_oop.core.my_gspread import GoogleTabs

import pandas as pd

In [ ]:
# Инициализируем базу данных и получаем данные о рекламных расходах
database = Database()
# Получаем данные о рекламных расходах по статьям за вчерашний день
adv_spend = database.read_sql_to_dataframe(query_adv_spend)
# Задаем параметры для работы с Google Sheets
table_title = unit_gs.get("title")
sheet_title = unit_gs.get("unit_sheet")
# Инициализируем работу с Google Sheets
google_tabs = GoogleTabs(table_title, sheet_title)
# Получаем данные из Google Sheets и преобразуем их в DataFrame
sheet_data = google_tabs.sheet_title.get_all_values()
headers = sheet_data[0]
rows = sheet_data[1:]
df = pd.DataFrame(rows, columns=headers)
# Приводим столбец 'Артикул' к типу int для корректного сравнения
df['Артикул'] = df['Артикул'].astype(int)
# Создаем датафрейм с нужными столбцами для дальнейшей работы
df_short = df[['Артикул', 'Реклама']]

# 1. Получаем список всех уникальных артикулов, по которым БЫЛИ затраты
articles_with_spend = set(adv_spend['article_id'].astype(int))

# 2. Создаем функцию для проверки
def check_adv(article):
    # Приводим к строке для надежности сравнения
    if int(article) in articles_with_spend:
        return "реклама"
    else:
        return ""

# 3. Применяем функцию к колонке 'Артикул' в df_short
df_short['Реклама'] = df_short['Артикул'].apply(check_adv)

# Динамически находим индекс колонки "Артикул", чтобы не ошибиться
if 'Артикул' not in headers:
    print("Ошибка: В таблице нет колонки 'Артикул'!")
else:
    art_idx = headers.index('Артикул')

    # --- 3. Сопоставление ---
    # Важно: мы создаем список значений в том же порядке, в котором идут строки в таблице
    results_list = []
    for row in rows:
        # Проверяем артикул из текущей строки таблицы
        current_art = int(row[art_idx])
        
        if current_art in articles_with_spend:
            results_list.append("реклама")
        else:
            results_list.append("")

    # --- 4. Запись ---
    # Теперь нам всё равно, где находится колонка "Реклама", метод сам её найдет
    google_tabs.update_column_by_name("Реклама", results_list)


✅ Успешное подключение к UNIT 2.0 (tested) -> MAIN (tested)


C:\Users\123\AppData\Local\Temp\ipykernel_19784\4207690136.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_short['Реклама'] = df_short['Артикул'].apply(check_adv)


✅ Данные успешно записаны в колонку 'Реклама' (диапазон E2:E7285)


In [6]:
# 1. Получаем список всех уникальных артикулов, по которым БЫЛИ затраты
articles_with_spend = set(adv_spend['article_id'].astype(int))

# 2. Создаем функцию для проверки
def check_adv(article):
    # Приводим к строке для надежности сравнения
    if int(article) in articles_with_spend:
        return "реклама"
    else:
        return ""

# 3. Применяем функцию к колонке 'Артикул' в df_short
df_short['Реклама'] = df_short['Артикул'].apply(check_adv)


# Динамически находим индекс колонки "Артикул", чтобы не ошибиться
if 'Артикул' not in headers:
    print("Ошибка: В таблице нет колонки 'Артикул'!")
else:
    art_idx = headers.index('Артикул')

    # --- 3. Сопоставление ---
    # Важно: мы создаем список значений в том же порядке, в котором идут строки в таблице
    results_list = []
    for row in rows:
        # Проверяем артикул из текущей строки таблицы
        current_art = int(row[art_idx])
        
        if current_art in articles_with_spend:
            results_list.append("реклама")
        else:
            results_list.append("")

    # --- 4. Запись ---
    # Теперь нам всё равно, где находится колонка "Реклама", метод сам её найдет
    google_tabs.update_column_by_name("Реклама", results_list)

C:\Users\123\AppData\Local\Temp\ipykernel_19784\3414333502.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_short['Реклама'] = df_short['Артикул'].apply(check_adv)


✅ Данные успешно записаны в колонку 'Реклама' (диапазон E2:E7285)


In [12]:
query = """
   SELECT a.nm_id,
       a.account,
       a.vendor_code,
       a.local_vendor_code,
       DATE(a.created_at) AS date,
       AVG(c.purchase_price) AS purchase_price
FROM article a
LEFT JOIN cost_price c
    ON c.local_vendor_code = a.local_vendor_code
    AND c.date = DATE(a.created_at)
WHERE  a.created_at >= '2026-03-31 16:26:19.010 +0300'
GROUP BY a.nm_id,
       a.account,
       a.vendor_code,
       a.local_vendor_code,
       DATE(a.created_at);
       """
analyze = FinReportsAnalyze()
df = analyze.get_df_from_db(query)

In [13]:
df.head()

,nm_id,account,vendor_code,local_vendor_code,date,purchase_price
0,928622213,ВЕКТОР5531,wild1289,wild1289,2026-04-01,786.0
1,928640495,ВЕКТОР5531,wild1247d,wild1247,2026-04-01,446.0
2,928611761,СТАРТ7409,wild164,wild164,2026-04-01,155.0
3,932608025,ВЕКТОР5531,wild1179,wild1179,2026-04-02,8008.0
4,932604242,СТАРТ7409,wild387,wild387,2026-04-02,3733.0


In [21]:
filtered_df = df[~(df['local_vendor_code'] != df['vendor_code']) & ~(df['local_vendor_code'].str.contains('wild', na=False))]
filtered_df.shape

(16, 6)

In [20]:
path = "C:\\Users\\123\\Downloads\\filtered_data.csv"
filtered_df.to_csv(path, index=False)